In [ ]:
#Import and install necessary packages
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
!pip install pycountry
import pycountry
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 46.0 MB/s eta 0:00:00


In [ ]:
BASE_URL = "https://api.worldbank.org/v2/indicator"

def fetch_all_indicators():
    records = []
    page = 1
    while True:
        params = {"format": "json", "source": 2, "per_page": 1000, "page": page}
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        meta, data = resp.json()[0], resp.json()[1]

        if not data:
            break

        for row in data:
            records.append({
                "code": row["id"],
                "name": row["name"],
                "source_note": (row.get("sourceNote") or "")[:150],
            })

        if page >= meta["pages"]:
            break
        page += 1
        time.sleep(0.2)

    return pd.DataFrame(records)

indicator_catalog = fetch_all_indicators()
print(indicator_catalog.shape)
indicator_catalog.head()

(1498, 3)


,code,name,source_note
0,AG.CON.FERT.PT.ZS,Fertilizer consumption (% of fertilizer produc...,Fertilizer consumption measures the quantity o...
1,AG.CON.FERT.ZS,Fertilizer consumption (kilograms per hectare ...,Fertilizer consumption measures the quantity o...
2,AG.LND.AGRI.K2,Agricultural land (sq. km),Agricultural land refers to the land area that...
3,AG.LND.AGRI.ZS,Agricultural land (% of land area),Agricultural land refers to the share of land ...
4,AG.LND.ARBL.HA,Arable land (hectares),Arable land (in hectares) includes land define...


In [ ]:
def find_indicator(keyword):
    matches = indicator_catalog[indicator_catalog["name"].str.contains(keyword, case=False, na=False)]
    return matches[["code", "name"]]

print(find_indicator("GDP per capita"))
print(find_indicator("health expenditure"))
print(find_indicator("immunization, dpt"))
print(find_indicator("basic sanitation"))
print(find_indicator("literacy rate, adult female"))

                  code                                               name
731     NY.GDP.PCAP.CD                       GDP per capita (current US$)
732     NY.GDP.PCAP.CN                       GDP per capita (current LCU)
733     NY.GDP.PCAP.KD                 GDP per capita (constant 2015 US$)
734  NY.GDP.PCAP.KD.ZG                   GDP per capita growth (annual %)
735     NY.GDP.PCAP.KN                      GDP per capita (constant LCU)
736  NY.GDP.PCAP.PP.CD      GDP per capita, PPP (current international $)
737  NY.GDP.PCAP.PP.KD  GDP per capita, PPP (constant 2021 internation...
956  SE.XPD.PRIM.PC.ZS  Government expenditure per student, primary (%...
958  SE.XPD.SECO.PC.ZS  Government expenditure per student, secondary ...
960  SE.XPD.TERT.PC.ZS  Government expenditure per student, tertiary (...
                     code                                               name
1107    SH.XPD.CHEX.GD.ZS              Current health expenditure (% of GDP)
1108    SH.XPD.CHEX.PC.CD  Curre

In [ ]:
#
BASE_URL = "https://api.worldbank.org/v2/country/all/indicator/{code}"

INDICATORS = {
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "SH.XPD.CHEX.GD.ZS": "health_exp_pct_gdp",
    "SH.IMM.IDPT": "dtp3_immunization_pct",
    "SH.STA.BASS.ZS": "basic_sanitation_pct",
    "SE.ADT.LITR.FE.ZS": "female_literacy_pct",
}

START_YEAR = 2021
END_YEAR = 2024


def fetch_indicator(code: str, colname: str) -> pd.DataFrame:
    all_records = []
    page = 1

    while True:
        params = {
            "format": "json",
            "date": f"{START_YEAR}:{END_YEAR}",
            "per_page": 1000,
            "page": page,
        }
        resp = requests.get(BASE_URL.format(code=code), params=params, timeout=30)
        resp.raise_for_status()
        meta, data = resp.json()[0], resp.json()[1]

        if not data:
            break

        for row in data:
            all_records.append({
                "country": row["country"]["value"],
                "country_code": row["countryiso3code"],
                "year": int(row["date"]),
                colname: row["value"],
            })

        if page >= meta["pages"]:
            break
        page += 1
        time.sleep(0.2)

    return pd.DataFrame(all_records)


def pull_all_indicators() -> pd.DataFrame:
    merged = None
    for code, colname in INDICATORS.items():
        print(f"Fetching {colname} ({code})...")
        df = fetch_indicator(code, colname)
        merged = df if merged is None else merged.merge(
            df, on=["country", "country_code", "year"], how="outer"
        )
    return merged


def coverage_by_year(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each year, count how many countries have NON-NULL values for
    ALL 5 indicators at once (not just each indicator separately) —
    that's what actually matters for the regression, since a row with
    any missing predictor can't be used as-is.
    """
    value_cols = list(INDICATORS.values())
    df["all_present"] = df[value_cols].notna().all(axis=1)

    summary = df.groupby("year")["all_present"].sum().reset_index()
    summary.columns = ["year", "countries_with_full_data"]
    return summary.sort_values("countries_with_full_data", ascending=False)


if __name__ == "__main__":
    wdi_df = pull_all_indicators()
    wdi_df.to_csv("wdi_2021_2024_raw.csv", index=False)
    print(f"\nPulled: {wdi_df.shape[0]} rows, {wdi_df.shape[1]} columns")

    coverage = coverage_by_year(wdi_df)
    print("\nCountries with COMPLETE data (all 5 indicators) per year:")
    print(coverage)

Fetching gdp_per_capita (NY.GDP.PCAP.CD)...
Fetching health_exp_pct_gdp (SH.XPD.CHEX.GD.ZS)...
Fetching dtp3_immunization_pct (SH.IMM.IDPT)...
Fetching basic_sanitation_pct (SH.STA.BASS.ZS)...
Fetching female_literacy_pct (SE.ADT.LITR.FE.ZS)...

Pulled: 1060 rows, 8 columns

Countries with COMPLETE data (all 5 indicators) per year:
   year  countries_with_full_data
1  2022                        86
0  2021                        83
2  2023                        70
3  2024                         1


In [ ]:
print(wdi_df.shape)   # should be (86, 7
print(wdi_df.head())
wdi_2022 = wdi_df[wdi_df['year'] == 2022]
print(wdi_2022.shape)
wdi_2022.head()

(1060, 9)
                       country country_code  year  gdp_per_capita  \
0                  Afghanistan          AFG  2021      356.496214   
1                  Afghanistan          AFG  2022      357.261153   
2                  Afghanistan          AFG  2023      413.757895   
3                  Afghanistan          AFG  2024      416.871146   
4  Africa Eastern and Southern          AFE  2021     1560.894626   

   health_exp_pct_gdp  dtp3_immunization_pct  basic_sanitation_pct  \
0           21.508444              55.000000             50.296640   
1           23.088169              58.000000             51.691440   
2           14.985763              60.000000             53.090937   
3                 NaN              59.000000             54.495799   
4            5.946320              74.627446             33.571066   

   female_literacy_pct  all_present  
0            22.600000         True  
1            26.600000         True  
2                  NaN        False  
3 

,country,country_code,year,gdp_per_capita,health_exp_pct_gdp,dtp3_immunization_pct,basic_sanitation_pct,female_literacy_pct,all_present
1,Afghanistan,AFG,2022,357.261153,23.088169,58.000000,51.691440,26.600000,True
5,Africa Eastern and Southern,AFE,2022,1675.902524,5.566469,74.689518,34.279170,68.010002,True
9,Africa Western and Central,AFW,2022,2143.072094,4.174501,69.692246,39.220504,52.470001,True
13,Albania,ALB,2022,7756.961887,7.536462,97.000000,99.299268,NaN,False
17,Algeria,DZA,2022,4960.303343,3.634643,77.000000,85.826286,NaN,False


In [ ]:
url = "https://ourworldindata.org/grapher/under-5-mortality-rate-sdgs.csv?v=1&csvType=full&useColumnShortNames=true"
mortality_under_5 = pd.read_csv(url, storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
#print(mortality_under_5.head())

#The mortality rate was collected in per 100, the rate was coverted to per 1000 by multipying per 100 to work with the general recorded rate of per 1000
mortality_under_5["per_1000"] = mortality_under_5["observation_value__indicator_child_mortality_rate__sex_total__wealth_quintile_total__unit_of_measure_deaths_per_100_live_births"] * 10
#mortality_under_5.head()

#The name of countries and their ISO code in existence
countries = []
for c in pycountry.countries:
    countries.append({
        'country_name': c.name,
        'iso_alpha3': c.alpha_3
    })

iso_data = pd.DataFrame(countries).sort_values('country_name').reset_index(drop=True)
#print(iso_data.head())

#Rename the column that contains the code for each country of iso data, so it can be merge with under_5 mortality data
iso_data = iso_data.rename(columns={'iso_alpha3': 'code', "country_name":"entity"})
#iso_data.head()

#Merge under_5_mertality data on the left with iso_data so as to filter the data to all country,
#because the under_5_mortality data as some world back indicator in between which does not belong to any country
mortality_under_5_new = pd.merge(mortality_under_5, iso_data, on=['code', "entity"], how='left')
#mortality_under_5_new.head()

#Validating my new data
#print(mortality_under_5.shape)
#print(mortality_under_5_new.shape)

#Then merge isn't the proper method, i will try filter method
mortality_under_5_new_2 = mortality_under_5[mortality_under_5["code"].isin(iso_data["code"])]
#print(mortality_under_5_new_2.shape)
#mortality_under_5_new_2.head()

#validating if we have all countries in the list
#print(mortality_under_5_new_2["code"].unique())
#unique_coutries = mortality_under_5_new_2["code"].unique()
#print(len(unique_coutries))

# Pull country metadata including income classification
url = "https://api.worldbank.org/v2/country?format=json&per_page=300"
response = requests.get(url)
data = response.json()[1]
# index 0 is pagination info, index 1 is the actual data

# Flatten the nested structure
income_level = pd.DataFrame([
    {
        "Code": c["id"],
        "country_name": c["name"],
        "income_group": c["incomeLevel"]["value"],
        "region": c["region"]["value"]
    }
    for c in data
])

# Drop non-country aggregates (World Bank includes regional groupings here too)
income_level = income_level[income_level["income_group"] != "Aggregates"]

#print(income_level["income_group"].unique())
#income_level.head()

#Now i have my income level data, so i will like extract two columns from it to the mortarlity data
mortality_under_5_new_2["income_level"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Code')['income_group'])
mortality_under_5_new_2["regions"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Code')['region'])
#print(mortality_under_5_new_2.head())
#print(mortality_under_5_new_2["income_level"].unique())
#print(mortality_under_5_new_2["regions"].unique())


#I waant to drop missing value in the data
mort_und_5_new_2_dropna = mortality_under_5_new_2.dropna()
#print(mort_und_5_new_2_dropna.shape)
#print(mort_und_5_new_2_dropna.isna().sum())

#Since my interest is working on years from year 2000 to later year.
mort_und_5_new_2_dropna_2000 = mort_und_5_new_2_dropna[mort_und_5_new_2_dropna["year"] >= 2000]
#print(mort_und_5_new_2_dropna_2000.shape)

#validating which countries have dropped since i applied dropping missing valuea nd also
#applied filtered for years above 2000
#print(mort_und_5_new_2_dropna_2000["code"].unique())
unique_coutries_2 = mort_und_5_new_2_dropna_2000["code"].unique()
#print(len(unique_coutries_2))

# Rename data
mortality_under_5 = mort_und_5_new_2_dropna_2000
mortality_under_5.head()

/tmp/ipykernel_753/4092836923.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_under_5_new_2["income_level"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Code')['income_group'])
/tmp/ipykernel_753/4092836923.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_under_5_new_2["regions"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Cod

,entity,code,year,observation_value__indicator_child_mortality_rate__sex_total__wealth_quintile_total__unit_of_measure_deaths_per_100_live_births,per_1000,income_level,regions
43,Afghanistan,AFG,2000,13.170724,131.70724,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
44,Afghanistan,AFG,2001,12.747986,127.47986,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
45,Afghanistan,AFG,2002,12.307344,123.07344,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
46,Afghanistan,AFG,2003,11.854750,118.54750,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
47,Afghanistan,AFG,2004,11.394402,113.94402,Low income,"Middle East, North Africa, Afghanistan & Pakistan"


In [ ]:
mortality_under_5.columns

Index(['entity', 'code', 'year',
       'observation_value__indicator_child_mortality_rate__sex_total__wealth_quintile_total__unit_of_measure_deaths_per_100_live_births',
       'per_1000', 'income_level', 'regions'],
      dtype='object')

In [ ]:
#Drop per_100
mortality_under_5 = mortality_under_5.drop(['observation_value__indicator_child_mortality_rate__sex_total__wealth_quintile_total__unit_of_measure_deaths_per_100_live_births'], axis=1)
mortality_under_5.head()

,entity,code,year,per_1000,income_level,regions
43,Afghanistan,AFG,2000,131.70724,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
44,Afghanistan,AFG,2001,127.47986,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
45,Afghanistan,AFG,2002,123.07344,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
46,Afghanistan,AFG,2003,118.54750,Low income,"Middle East, North Africa, Afghanistan & Pakistan"
47,Afghanistan,AFG,2004,113.94402,Low income,"Middle East, North Africa, Afghanistan & Pakistan"


In [ ]:
#rename columnns for easy merging
wdi_2022 = wdi_2022.rename(columns={'country_code': 'code', "country":"entity"})
print(wdi_2022.head())
print(wdi_2022.columns)
print(mortality_under_5.columns)

                         entity code  year  gdp_per_capita  \
1                   Afghanistan  AFG  2022      357.261153   
5   Africa Eastern and Southern  AFE  2022     1675.902524   
9    Africa Western and Central  AFW  2022     2143.072094   
13                      Albania  ALB  2022     7756.961887   
17                      Algeria  DZA  2022     4960.303343   

    health_exp_pct_gdp  dtp3_immunization_pct  basic_sanitation_pct  \
1            23.088169              58.000000             51.691440   
5             5.566469              74.689518             34.279170   
9             4.174501              69.692246             39.220504   
13            7.536462              97.000000             99.299268   
17            3.634643              77.000000             85.826286   

    female_literacy_pct  all_present  
1             26.600000         True  
5             68.010002         True  
9             52.470001         True  
13                  NaN        False  
17   

In [ ]:
#Marge the mortality data with the socio economic indicator from world ban
mortality_socio_22 = pd.merge(mortality_under_5, wdi_2022, on=['code', "entity", "year"])
mortality_socio_22.head()

,entity,code,year,per_1000,income_level,regions,gdp_per_capita,health_exp_pct_gdp,dtp3_immunization_pct,basic_sanitation_pct,female_literacy_pct,all_present
0,Afghanistan,AFG,2022,56.770400,Low income,"Middle East, North Africa, Afghanistan & Pakistan",357.261153,23.088169,58.0,51.691440,26.6,True
1,Albania,ALB,2022,9.436385,Upper middle income,Europe & Central Asia,7756.961887,7.536462,97.0,99.299268,NaN,False
2,Algeria,DZA,2022,22.284396,Upper middle income,"Middle East, North Africa, Afghanistan & Pakistan",4960.303343,3.634643,77.0,85.826286,NaN,False
3,Andorra,AND,2022,2.685471,High income,Europe & Central Asia,42414.059011,7.521358,98.0,99.999996,NaN,False
4,Angola,AGO,2022,51.863985,Lower middle income,Sub-Saharan Africa,3598.536691,2.600410,54.0,50.295136,NaN,False


In [ ]:
mortality_socio_22.to_csv("/content/drive/MyDrive/Portfolio/30 Days Challenge/Day 8/mortality_socio_22.csv")

In [ ]:
#Validate and check for missing value
print(mortality_socio_22.shape)
print(mortality_socio_22.isna().sum())

(168, 12)
entity                     0
code                       0
year                       0
per_1000                   0
income_level               0
regions                    0
gdp_per_capita             4
health_exp_pct_gdp         3
dtp3_immunization_pct      2
basic_sanitation_pct       8
female_literacy_pct      126
all_present                0
dtype: int64


In [ ]:
mortality_socio_22.columns

Index(['entity', 'code', 'year', 'per_1000', 'income_level', 'regions',
       'gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct',
       'basic_sanitation_pct', 'female_literacy_pct', 'all_present'],
      dtype='object')

In [ ]:
mortality_socio_22_non_f = mortality_socio_22[['entity', 'code', 'year', 'per_1000', 'income_level', 'regions',
                                               'gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct','basic_sanitation_pct']]
mortality_socio_22_non_f.head()

,entity,code,year,per_1000,income_level,regions,gdp_per_capita,health_exp_pct_gdp,dtp3_immunization_pct,basic_sanitation_pct
0,Afghanistan,AFG,2022,56.770400,Low income,"Middle East, North Africa, Afghanistan & Pakistan",357.261153,23.088169,58.0,51.691440
1,Albania,ALB,2022,9.436385,Upper middle income,Europe & Central Asia,7756.961887,7.536462,97.0,99.299268
2,Algeria,DZA,2022,22.284396,Upper middle income,"Middle East, North Africa, Afghanistan & Pakistan",4960.303343,3.634643,77.0,85.826286
3,Andorra,AND,2022,2.685471,High income,Europe & Central Asia,42414.059011,7.521358,98.0,99.999996
4,Angola,AGO,2022,51.863985,Lower middle income,Sub-Saharan Africa,3598.536691,2.600410,54.0,50.295136


In [ ]:
mortality_socio_22_non_f.isna().sum()

,0
entity,0
code,0
year,0
per_1000,0
income_level,0
regions,0
gdp_per_capita,4
health_exp_pct_gdp,3
dtp3_immunization_pct,2
basic_sanitation_pct,8


In [ ]:
# Fill missing values using the median within each income_level group
cols_to_impute = ['gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct', 'basic_sanitation_pct']

for col in cols_to_impute:
    mortality_socio_22_non_f[col] = mortality_socio_22_non_f.groupby('income_level')[col].transform(lambda x: x.fillna(x.median()))

# Check nothing's left missing
print(mortality_socio_22_non_f[cols_to_impute].isna().sum())

gdp_per_capita           0
health_exp_pct_gdp       0
dtp3_immunization_pct    0
basic_sanitation_pct     0
dtype: int64


/tmp/ipykernel_753/1860671551.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_socio_22_non_f[col] = mortality_socio_22_non_f.groupby('income_level')[col].transform(lambda x: x.fillna(x.median()))
/tmp/ipykernel_753/1860671551.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_socio_22_non_f[col] = mortality_socio_22_non_f.groupby('income_level')[col].transform(lambda x: x.fillna(x.median()))
/tmp/ipykernel_753/1860671551.py:5: SettingWithCopyWarning: 
A value is trying to be set on 

In [ ]:
print(mortality_socio_22_non_f.columns)
print(mortality_socio_22_non_f.shape)

Index(['entity', 'code', 'year', 'per_1000', 'income_level', 'regions',
       'gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct',
       'basic_sanitation_pct'],
      dtype='object')
(168, 10)


In [ ]:
mortality_socio_22_non_f.to_csv("/content/drive/MyDrive/Portfolio/30 Days Challenge/Day 8/mortality_socio_22_non_f.csv")